# Coupon Experiment Decision Framework — Walkthrough

This notebook walks through the framework stage by stage using simulated data.

Note: the dataset contains hidden latent fields for later validation. Stages 1-4 only use observable fields; latent truth and long-term outcomes are revealed in Stage 5.

> **Data note:** The simulated dataset includes latent fields for validation
> purposes. The walkthrough treats those fields as unavailable until the
> calibration stage. Stages 1 to 4 use only observable fields.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/simulated_coupon_experiment.csv")

# Stages 1-4 only use observable fields. The CSV also contains latent ground
# truth and long-term outcome columns, which are intentionally not loaded or
# shown until Stage 5.
observable_cols = [
    "user_id",
    "group",
    "coupon_received",
    "signup_day",
    "exposure_day",
    "made_first_purchase_30d",
    "days_to_first_purchase",
    "repeat_visits_14d",
    "browse_sessions_14d",
    "product_views_14d",
    "cart_adds_14d",
    "discount_page_views_14d",
    "coupon_used",
]

df[observable_cols].head()

## Stage 1: Measurement Design

Stage 1 is about designing the experiment so it can answer a stronger question than "did the 30-day first-purchase rate go up?" — it should also capture what *kind* of value any lift represents. That means committing up front to the primary metric, guardrail metrics, holdout logic, observation windows, and the post-period tracking needed to later assess incremental demand, customer quality, and borrowed-demand risk.

### Trap 1: The Headline Lift

In [ ]:
rates = (
    df.groupby("group")["made_first_purchase_30d"]
    .mean()
    .rename("first_purchase_rate_30d")
)

control_rate = rates.loc["control"]
treatment_rate = rates.loc["treatment"]
lift_pp = (treatment_rate - control_rate) * 100

print(f"Control   30d first-purchase rate: {control_rate:.2%}")
print(f"Treatment 30d first-purchase rate: {treatment_rate:.2%}")
print(f"Headline lift:                     {lift_pp:+.2f}pp")

rates.to_frame()

The coupon appears to improve the 30-day first-purchase rate. But this only answers whether the rate went up. It does not tell us what kind of value that lift represents.

### Trap 2: Same Lift, Different Reality

| Scenario | Headline lift | Dominant hidden reality | Decision implication |
|----------|---------------|-------------------------|----------------------|
| A | +6pp | More persuadable high-value users | Consider scale |
| B | +6pp | More discount-driven and pulled-forward users | Adjust or stop |

Two experiments can show the same headline lift while representing very different underlying realities. The same +6pp could justify scaling in one case and stopping in another. This is the gap the rest of the framework aims to close. We will return to validate this with long-term outcomes in Stage 5.

## Stage 2: Early Signal Layer

While long-term outcomes are still immature, we turn raw early behaviour into
a small set of transparent, rule-based **derived signals**. Each signal is a
simple combination of *observable* fields only, expressed on a common 0–1
percentile scale so that fields with different units stay comparable:

- **engagement_intensity** — breadth of early activity (repeat visits, browse
  sessions, product views).
- **purchase_intent** — depth of intent (cart adds), kept separate so product
  views are not weighted twice.
- **discount_dependency** — reliance on the discount itself (discount-page
  views combined with whether a coupon was redeemed).
- **early_timing** — how early the first purchase landed (purchasers only; NaN
  otherwise). On its own this is *not* a borrowed-demand signal.

In [ ]:
import sys

sys.path.insert(0, "../src")
from signals import add_derived_signals

# Stage 2 reads observable fields only.
signals_df = add_derived_signals(df[observable_cols])

derived_cols = [
    "engagement_intensity",
    "purchase_intent",
    "discount_dependency",
    "early_timing",
]

# Show the observable inputs alongside the derived signals only - latent and
# long-term columns are never loaded here.
signals_df[["user_id", "group", "made_first_purchase_30d"] + derived_cols].head()

Each derived signal is a percentile within the cohort, so it reads as *where a
user sits relative to everyone else* rather than a raw count. High
`engagement_intensity` and `purchase_intent` point toward genuine interest;
high `discount_dependency` points toward discount-led behaviour; `early_timing`
simply flags an early purchase, which only becomes meaningful **in combination**
with the other signals in Stage 3.

## Stage 3: Early Read Model

Stage 3 turns the early signals into an initial read on customer quality and
borrowed-demand risk. This component is **intentionally modular**: the v1 below
is a transparent, rule-based read, and it could later be swapped for a
statistical or ML model without changing the rest of the framework, as long as
it emits the same read columns.

It follows a "score, then band" shape. Continuous scores are computed
internally as an intermediate step, then exposed as a coarse **high / medium /
low** read using within-cohort tertiles — no hand-picked thresholds, and no
false precision.

- **quality_score** — high engagement + high intent + low discount dependency.
- **borrowed_risk_score** — early purchase **and** weak engagement **and** high
  discount dependency, combined as a product so *all three* must be present.
  Buying early on its own does not raise it. Purchasers only.
- **discount_dependency_score** — the `discount_dependency` signal directly.

In [ ]:
from early_read import add_early_read

read_df = add_early_read(signals_df)

score_cols = ["quality_score", "borrowed_risk_score", "discount_dependency_score"]
read_cols = ["quality_read", "borrowed_risk_read", "discount_dependency_read"]

# Internal scores are intermediate; the high/medium/low reads are what we act on.
read_df[["user_id", "group"] + score_cols + read_cols].head()

### Cohort-level Read

The read is summarised at the **cohort level**, not per user. We compare the
*mix* of high / medium / low reads among **purchasers** in each arm. This keeps
the output a decision-useful read on the cohort rather than an individual-level
label or prediction.

In [ ]:
purchasers = read_df[read_df["made_first_purchase_30d"] == 1]
levels = ["high", "medium", "low"]

parts = []
for col in read_cols:
    mix = (
        purchasers.groupby("group")[col]
        .value_counts(normalize=True)
        .unstack("group")
        .reindex(index=levels)
    )
    mix.index = pd.MultiIndex.from_product([[col], mix.index], names=["read", "level"])
    parts.append(mix)

# Share (%) of each arm's purchasers falling in each read band, plus the gap.
cohort_summary = pd.concat(parts) * 100
cohort_summary["delta_pp"] = cohort_summary["treatment"] - cohort_summary["control"]
cohort_summary.round(1)

**Reading the cohort.** Compared with control purchasers, treatment purchasers
show a much higher share of **high** discount-dependency reads (about 73% vs
21%) and **high** borrowed-risk reads (about 51% vs 11%), alongside a much
lower share of **high**-quality reads (about 36% vs 70%). Taken together, that
is a decision-useful read that a meaningful part of the headline lift may be
carried by discount-led, pulled-forward purchases rather than durable
incremental demand.

This stays at the cohort level on purpose. It is a read on the *mix* of the
treatment purchaser cohort — not a label on any individual user, and not a
prediction of any one user's long-term value. It points to where the risk
likely sits. Stage 4 turns this into a scale / adjust / stop recommendation,
and Stage 5 checks the read against mature long-term outcomes.